# 03 - CPI Surprise Probability Engine

## Objective

Build an unconditional CPI-surprise distribution from historical macro features,
map that distribution into Kalshi bucket probabilities, and compare model vs market.

## Inputs

- `../data/macro_features.parquet`
- `../data/kalshi_cpi_markets_open.parquet`
- `../data/kalshi_cpi_market_probs_snapshot.parquet`

## Outputs

- `../data/output/engine_results.parquet`
- `../data/output/summary.json`
- A bar chart of `p_model` vs `p_market` by bucket

In [7]:
# Debug: inspect actual columns in kalshi market metadata
import pandas as pd

mk = pd.read_parquet("../data/kalshi_cpi_markets_open.parquet")
print(mk.columns.tolist())
mk.head(3)

['ticker', 'event_ticker', 'title', 'subtitle', 'close_time', 'expected_expiration_time', 'expiration_time', 'strike_type', 'custom_strike', 'last_price', 'yes_bid_dollars', 'yes_ask_dollars', 'no_bid_dollars', 'no_ask_dollars', 'pulled_series', 'pulled_at_utc']


,ticker,event_ticker,title,subtitle,close_time,expected_expiration_time,expiration_time,strike_type,custom_strike,last_price,yes_bid_dollars,yes_ask_dollars,no_bid_dollars,no_ask_dollars,pulled_series,pulled_at_utc
0,KXECONSTATCPIYOY-26NOV-T3.5,KXECONSTATCPIYOY-26NOV,CPI year-over-year in Nov 2026?,,2026-12-10T13:29:00Z,2026-12-10T15:00:00Z,2027-03-11T15:00:00Z,custom,{'Value': '3.5'},0,0.0000,1.0000,0.0000,1.0000,KXECONSTATCPIYOY,2026-01-28T02:47:00.685725+00:00
1,KXECONSTATCPIYOY-26NOV-T3.4,KXECONSTATCPIYOY-26NOV,CPI year-over-year in Nov 2026?,,2026-12-10T13:29:00Z,2026-12-10T15:00:00Z,2027-03-11T15:00:00Z,custom,{'Value': '3.4'},0,0.0000,1.0000,0.0000,1.0000,KXECONSTATCPIYOY,2026-01-28T02:47:00.685725+00:00
2,KXECONSTATCPIYOY-26NOV-T3.3,KXECONSTATCPIYOY-26NOV,CPI year-over-year in Nov 2026?,,2026-12-10T13:29:00Z,2026-12-10T15:00:00Z,2027-03-11T15:00:00Z,custom,{'Value': '3.3'},0,0.0000,1.0000,0.0000,1.0000,KXECONSTATCPIYOY,2026-01-28T02:47:00.685725+00:00


In [9]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm

print(mk.columns.tolist())
print(mk.head(3))
# ------------------------------------------------------------------
# 1) Load upstream artifacts
# ------------------------------------------------------------------
macro = pd.read_parquet("../data/macro_features.parquet").copy()
mk = pd.read_parquet("../data/kalshi_cpi_markets_open.parquet").copy()
p_mkt = pd.read_parquet("../data/kalshi_cpi_market_probs_snapshot.parquet").copy()

print(f"macro rows: {len(macro):,}")
print(f"market rows (raw): {len(mk):,}")
print(f"market prob rows (raw): {len(p_mkt):,}")

# ------------------------------------------------------------------
# 2) Build YoY CPI level distribution for Kalshi bucket pricing
# mu = latest core_cpi_yoy (in % units)
# sigma = std of monthly YoY changes, annualized to a reasonable spread
# ------------------------------------------------------------------
if "core_cpi_yoy" not in macro.columns:
    raise KeyError("Expected 'core_cpi_yoy' in macro_features.parquet")

yoy = macro["core_cpi_yoy"].dropna().copy()
if yoy.empty:
    raise ValueError("core_cpi_yoy series is empty.")

# Convert to percent units if data is decimal-scale (e.g., 0.028 -> 2.8)
if yoy.abs().median() < 1.0:
    yoy = yoy * 100.0
    print("Scaled core_cpi_yoy by 100 to percent units.")

mu = float(yoy.iloc[-1])

yoy_mom_change = yoy.diff().dropna()
if yoy_mom_change.empty:
    sigma = 0.3
else:
    sigma_hist_annualized = float(yoy_mom_change.std() * np.sqrt(12))
    sigma = sigma_hist_annualized if sigma_hist_annualized > 0 else 0.3

# Keep a minimum spread so probabilities are meaningful across bucket ladder.
sigma = max(float(sigma), 0.3)

print(f"yoy observations: {len(yoy):,}")
print(f"fitted normal mu={mu:.6f}, sigma={sigma:.6f}")

# ------------------------------------------------------------------
# 3) Prepare Kalshi bucket definitions from custom_strike
# each ticker is a threshold; derive ranges from adjacent strikes
# ------------------------------------------------------------------
required_cols = ["ticker", "custom_strike", "strike_type"]
missing_required = [c for c in required_cols if c not in mk.columns]
if missing_required:
    raise KeyError(f"Missing required market columns: {missing_required}")

mk_buckets = mk.copy()
mk_buckets = mk_buckets[mk_buckets["strike_type"].astype(str).str.lower() == "custom"].copy()


def parse_custom_strike(x):
    if isinstance(x, dict):
        val = x.get("Value")
    elif isinstance(x, str):
        s = x.strip()
        if s.startswith("{") and "Value" in s:
            try:
                val = json.loads(s.replace("'", '"')).get("Value")
            except Exception:
                return np.nan
        else:
            val = s
    else:
        val = x
    return pd.to_numeric(val, errors="coerce")


mk_buckets["strike_value"] = mk_buckets["custom_strike"].apply(parse_custom_strike)
rows_before_drop = len(mk_buckets)
mk_buckets = mk_buckets.dropna(subset=["strike_value"]).copy()
mk_buckets = mk_buckets.sort_values(by=["event_ticker", "strike_value", "ticker"]).reset_index(drop=True)

# Derive floor/cap from adjacent strikes per event.
mk_buckets["floor"] = mk_buckets["strike_value"]
mk_buckets["cap"] = mk_buckets.groupby("event_ticker")["strike_value"].shift(-1)

# Lowest bucket starts at -inf. Highest bucket ends at +inf.
first_idx = mk_buckets.groupby("event_ticker").head(1).index
last_idx = mk_buckets.groupby("event_ticker").tail(1).index
mk_buckets.loc[first_idx, "floor"] = -np.inf
mk_buckets.loc[last_idx, "cap"] = np.inf

rows_after_drop = len(mk_buckets)
print(f"kept custom-strike rows: {rows_after_drop:,} / {rows_before_drop:,}")

# ------------------------------------------------------------------
# 4) Integrate fitted distribution over each bucket range
# ------------------------------------------------------------------
mk_buckets["p_model"] = np.where(
    np.isneginf(mk_buckets["floor"]),
    norm.cdf(mk_buckets["cap"], loc=mu, scale=sigma),
    np.where(
        np.isposinf(mk_buckets["cap"]),
        1.0 - norm.cdf(mk_buckets["floor"], loc=mu, scale=sigma),
        norm.cdf(mk_buckets["cap"], loc=mu, scale=sigma)
        - norm.cdf(mk_buckets["floor"], loc=mu, scale=sigma),
    ),
)
mk_buckets["p_model"] = np.clip(mk_buckets["p_model"], 0.0, 1.0)


def fmt_bound(v):
    if np.isneginf(v):
        return "-inf"
    if np.isposinf(v):
        return "+inf"
    return f"{v:g}"


mk_buckets["bucket_range"] = mk_buckets.apply(
    lambda r: f"{fmt_bound(r['floor'])} to {fmt_bound(r['cap'])}", axis=1
)

# ------------------------------------------------------------------
# 5) Join with market implied probabilities
# p_market may be stored in cents (e.g., 55) or dollars (0.55)
# ------------------------------------------------------------------
if "ticker" not in p_mkt.columns or "p_market" not in p_mkt.columns:
    raise KeyError("Expected columns ['ticker', 'p_market'] in kalshi_cpi_market_probs_snapshot.parquet")

p_mkt_clean = p_mkt[["ticker", "p_market"]].copy()
p_mkt_clean["p_market"] = pd.to_numeric(p_mkt_clean["p_market"], errors="coerce")

# If scale looks like cents, convert to probability dollars.
if p_mkt_clean["p_market"].dropna().gt(1).mean() > 0.5:
    p_mkt_clean["p_market"] = p_mkt_clean["p_market"] / 100.0

# Keep latest snapshot per ticker if duplicates exist.
p_mkt_clean = p_mkt_clean.drop_duplicates(subset=["ticker"], keep="last")

# Debug ticker coverage before merge.
mk_tickers = set(mk_buckets["ticker"].dropna().unique().tolist())
p_tickers = set(p_mkt_clean["ticker"].dropna().unique().tolist())
print("mk_buckets unique tickers:", len(mk_tickers))
print("p_mkt_clean unique tickers:", len(p_tickers))
print("sample mk_buckets tickers:", sorted(list(mk_tickers))[:10])
print("sample p_mkt_clean tickers:", sorted(list(p_tickers))[:10])

# If event cohorts differ (e.g., 26APR vs 26NOV), align both sides to one common event.
mk_buckets["event_code"] = mk_buckets["ticker"].str.extract(r"KXECONSTATCPIYOY-([^-]+)-", expand=False)
p_mkt_clean["event_code"] = p_mkt_clean["ticker"].str.extract(r"KXECONSTATCPIYOY-([^-]+)-", expand=False)

mk_events = set(mk_buckets["event_code"].dropna().unique().tolist())
p_events = set(p_mkt_clean["event_code"].dropna().unique().tolist())
common_events = mk_events.intersection(p_events)

print("mk_buckets events:", sorted(list(mk_events)))
print("p_mkt_clean events:", sorted(list(p_events)))
print("common events:", sorted(list(common_events)))

if common_events:
    p_event_counts = p_mkt_clean[p_mkt_clean["event_code"].isin(common_events)]["event_code"].value_counts()
    selected_event = p_event_counts.idxmax()
    mk_buckets = mk_buckets[mk_buckets["event_code"] == selected_event].copy()
    p_mkt_clean = p_mkt_clean[p_mkt_clean["event_code"] == selected_event].copy()
    print(f"Filtered both dataframes to event_code={selected_event}")

engine = mk_buckets[["ticker", "bucket_range", "p_model"]].merge(
    p_mkt_clean[["ticker", "p_market"]],
    on="ticker",
    how="left",
)
engine["edge"] = engine["p_model"] - engine["p_market"]

engine = engine[["ticker", "bucket_range", "p_model", "p_market", "edge"]].sort_values(
    by="ticker"
).reset_index(drop=True)

print(f"engine output rows: {len(engine):,}")
print(f"rows with market probs: {engine['p_market'].notna().sum():,}")
display(engine.head(15))

# ------------------------------------------------------------------
# 6) Save artifacts
# ------------------------------------------------------------------
os.makedirs("../data/output", exist_ok=True)

engine_out = "../data/output/engine_results.parquet"
summary_out = "../data/output/summary.json"

engine.to_parquet(engine_out, index=False)

summary = {
    "n_surprise_obs": int(len(yoy)),
    "normal_fit": {"mu": float(mu), "sigma": float(sigma)},
    "n_market_rows_raw": int(len(mk)),
    "n_market_rows_used": int(len(mk_buckets)),
    "n_engine_rows": int(len(engine)),
    "n_rows_with_p_market": int(engine["p_market"].notna().sum()),
    "p_model_sum": float(engine["p_model"].sum()),
    "p_market_mean": float(engine["p_market"].dropna().mean()) if engine["p_market"].notna().any() else None,
    "edge_mean": float(engine["edge"].dropna().mean()) if engine["edge"].notna().any() else None,
}

with open(summary_out, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"Saved: {engine_out}")
print(f"Saved: {summary_out}")
summary

# ------------------------------------------------------------------
# 7) Simple bar chart: p_model vs p_market by bucket
# ------------------------------------------------------------------
plot_df = engine.dropna(subset=["p_market"]).copy()
plot_df = plot_df.sort_values(by=["ticker"]).reset_index(drop=True)

if plot_df.empty:
    print("No rows with p_market available for plotting.")
else:
    x = np.arange(len(plot_df))
    width = 0.42

    plt.figure(figsize=(max(10, len(plot_df) * 0.3), 6))
    plt.bar(x - width / 2, plot_df["p_model"], width=width, label="p_model")
    plt.bar(x + width / 2, plot_df["p_market"], width=width, label="p_market")

    plt.xticks(x, plot_df["bucket_range"], rotation=90)
    plt.ylabel("Probability")
    plt.xlabel("Kalshi CPI bucket")
    plt.title("Model vs Market Probability by CPI Bucket")
    plt.legend()
    plt.tight_layout()
    plt.show()

['ticker', 'event_ticker', 'title', 'subtitle', 'close_time', 'expected_expiration_time', 'expiration_time', 'strike_type', 'custom_strike', 'last_price', 'yes_bid_dollars', 'yes_ask_dollars', 'no_bid_dollars', 'no_ask_dollars', 'pulled_series', 'pulled_at_utc']
                        ticker            event_ticker  \
0  KXECONSTATCPIYOY-26NOV-T3.5  KXECONSTATCPIYOY-26NOV   
1  KXECONSTATCPIYOY-26NOV-T3.4  KXECONSTATCPIYOY-26NOV   
2  KXECONSTATCPIYOY-26NOV-T3.3  KXECONSTATCPIYOY-26NOV   

                             title subtitle            close_time  \
0  CPI year-over-year in Nov 2026?           2026-12-10T13:29:00Z   
1  CPI year-over-year in Nov 2026?           2026-12-10T13:29:00Z   
2  CPI year-over-year in Nov 2026?           2026-12-10T13:29:00Z   

  expected_expiration_time       expiration_time strike_type  \
0     2026-12-10T15:00:00Z  2027-03-11T15:00:00Z      custom   
1     2026-12-10T15:00:00Z  2027-03-11T15:00:00Z      custom   
2     2026-12-10T15:00:00Z  2027-03

,ticker,bucket_range,p_model,p_market,edge
0,KXECONSTATCPIYOY-26NOV-T2.0,-inf to 2.1,1.000000e+00,NaN,NaN
1,KXECONSTATCPIYOY-26NOV-T2.1,2.1 to 2.2,1.167622e-12,NaN,NaN
2,KXECONSTATCPIYOY-26NOV-T2.2,2.2 to 2.3,1.034728e-13,NaN,NaN
3,KXECONSTATCPIYOY-26NOV-T2.3,2.3 to 2.4,8.104628e-15,NaN,NaN
4,KXECONSTATCPIYOY-26NOV-T2.4,2.4 to 2.5,6.661338e-16,NaN,NaN
5,KXECONSTATCPIYOY-26NOV-T2.5,2.5 to 2.6,0.000000e+00,NaN,NaN
6,KXECONSTATCPIYOY-26NOV-T2.6,2.6 to 2.7,0.000000e+00,NaN,NaN
7,KXECONSTATCPIYOY-26NOV-T2.7,2.7 to 2.8,0.000000e+00,NaN,NaN
8,KXECONSTATCPIYOY-26NOV-T2.8,2.8 to 2.9,0.000000e+00,NaN,NaN
9,KXECONSTATCPIYOY-26NOV-T2.9,2.9 to 3,0.000000e+00,NaN,NaN


Saved: ../data/output/engine_results.parquet
Saved: ../data/output/summary.json
No rows with p_market available for plotting.
